# FlyRank Capstone — Full Pipeline (Run Once, Download Results)

**Lane:** Refresh / Content Opportunity Scoring
**Question:** Which content items should be prioritized for refresh review, based only on search-performance signals observable at the time of the decision?

This single notebook does the entire capstone pipeline end to end:
data → decision-window features → future-outcome label → baseline → model →
validation (grouped split, leakage audit) → sealed test → ranked recommendations
→ small result files you download and send back.

**It does not export any raw warehouse rows** — only small aggregated JSON/CSV
result files, safe to share and commit.

**How to run:** Runtime → Run all. Needs your `HF_TOKEN` set in Colab secrets
(the key icon in the left sidebar), same as your ML-04 notebook already used.


In [ ]:
!pip -q install duckdb fsspec huggingface_hub scikit-learn

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF token loaded:", bool(HF_TOKEN))

OUT_DIR = Path("/content/capstone_outputs")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"""
INSTALL httpfs;
LOAD httpfs;
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")
print("Connected to warehouse.")

## Helper functions

Same decision-window / outcome-window contract your ML-04 notebook already
established: features are aggregated over a single "decision month," the
label is built from a separate "outcome month," and the two never mix.

In [ ]:
WAREHOUSE_BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

FEATURE_COLUMNS = [
    "impressions", "clicks", "avg_position", "ctr",
    "ga4_sessions", "sessions_ai", "scroll_events", "days_active",
]

def month_glob(month):
    return f"{WAREHOUSE_BASE}/month={month}/*.parquet"

def aggregate_month_features(month):
    query = f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE NULL END AS ctr,
        SUM(COALESCE(ga4_sessions, 0)) AS ga4_sessions,
        SUM(COALESCE(sessions_ai, 0)) AS sessions_ai,
        SUM(COALESCE(scroll_events, 0)) AS scroll_events,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE OR ga4_data_available IS TRUE) AS days_active
    FROM read_parquet('{month_glob(month)}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    """
    return con.execute(query).df()

def aggregate_month_outcome(month, metric="gsc_impressions"):
    query = f"""
    SELECT client_hash_id, content_hash_id, SUM({metric}) AS outcome_value
    FROM read_parquet('{month_glob(month)}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    """
    return con.execute(query).df()

def build_dev_frame(decision_month, outcome_month):
    features = aggregate_month_features(decision_month)
    outcome = aggregate_month_outcome(outcome_month).rename(columns={"outcome_value": "outcome_impressions"})
    frame = features.merge(outcome, on=["client_hash_id", "content_hash_id"], how="inner")
    frame["future_decline"] = (frame["outcome_impressions"] < frame["impressions"]).astype(int)
    return frame.drop(columns=["outcome_impressions"])

def group_holdout_split(df, group_col, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    groups = np.array(df[group_col].dropna().unique(), dtype=object)
    rng.shuffle(groups)
    n_test = max(1, int(len(groups) * test_size))
    test_groups = set(groups[:n_test])
    is_test = df[group_col].isin(test_groups)
    return df.loc[~is_test].copy(), df.loc[is_test].copy()

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

def base_rate(y_true):
    v = pd.Series(list(y_true))
    return float(v.mean()) if len(v) else 0.0

def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, sort_keys=True, default=str))

def escape_xml(v):
    return (str(v).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
            .replace('"',"&quot;").replace("'","&apos;"))

def svg_bar_chart(title, labels, values, path, caption="", color="#6F4E7C"):
    labels = [str(l) for l in labels]
    values = [float(v) if np.isfinite(v) else 0.0 for v in values]
    width, height = 960, 520
    max_v = max(max(values, default=1), 1)
    ml, mr, mt = 220, 40, 70
    mb = 70 if caption else 40
    pw, ph = width-ml-mr, height-mt-mb
    gap = 10
    bh = max(14, (ph - gap*max(len(values)-1,0)) / max(len(values),1))
    lines = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             '<rect width="100%" height="100%" fill="#ffffff"/>',
             f'<text x="{width/2}" y="34" text-anchor="middle" font-family="Arial" font-size="22" fill="#16232a">{escape_xml(title)}</text>']
    for i,(l,v) in enumerate(zip(labels, values)):
        y = mt + i*(bh+gap)
        bw = (v/max_v)*pw
        lines.append(f'<text x="{ml-12}" y="{y+bh*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{escape_xml(l[:38])}</text>')
        lines.append(f'<rect x="{ml}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" fill="{color}" rx="4"/>')
        lines.append(f'<text x="{ml+bw+8:.1f}" y="{y+bh*0.65:.1f}" font-family="Arial" font-size="13" fill="#27343b">{v:,.3g}</text>')
    if caption:
        lines.append(f'<text x="{ml}" y="{height-20}" font-family="Arial" font-size="12" fill="#5a6a72">{escape_xml(caption[:160])}</text>')
    lines.append("</svg>")
    Path(path).write_text("\n".join(lines))

print("Helpers ready.")

## 1. Data contract — verify grain, availability, missingness (ML-04 style)

Quick sanity checks before anything else — same checks your ML-04 notebook ran.

In [ ]:
check_month = "2026-03"
q1 = con.execute(f"""
SELECT COUNT(*) row_count, MIN(report_date) first_date, MAX(report_date) last_date
FROM read_parquet('{month_glob(check_month)}')
""").df()
print("March 2026 row count / date range:")
print(q1)

q2 = con.execute(f"""
SELECT COUNT(*) total,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) gsc_avail,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) ga4_avail
FROM read_parquet('{month_glob(check_month)}')
""").df()
print("\nAvailability:")
print(q2)

## 2. Build the decision → outcome frames

- **Dev frame:** March 2026 (decision/features) → April 2026 (outcome/label). Used for baseline + model development.
- **Sealed test frame:** May 2026 (decision) → June 2026 (outcome). Touched exactly once, at the end.
- **Live scoring frame:** June 2026 (decision only, no label — July isn't in the warehouse yet). This produces the actual forward-looking recommendation queue.


In [ ]:
dev_frame = build_dev_frame("2026-03", "2026-04")
print("Dev frame rows:", len(dev_frame), " | base rate (future_decline):", round(base_rate(dev_frame["future_decline"]), 4))
dev_frame.to_csv(OUT_DIR / "dev_frame_mar_apr.csv", index=False)
dev_frame.head()

## 3. Baseline — transparent rule + reason codes

**The rule, in plain words:** a content item is worth flagging for refresh review
if it already earns real search visibility but is wasting it — either sitting
off page one (weak average position) or converting that visibility into clicks
worse than a typical visible item (low CTR). Visibility without payoff is the
signal.

**Reason codes:** `visible_weak_position`, `visible_low_ctr`,
`general_refresh_review` (visible but neither specific flag tripped),
`not_visible_enough` (too little traffic to judge either way — monitor only).


In [ ]:
df = dev_frame.copy()
IMPRESSIONS_VISIBLE = df["impressions"].quantile(0.50)
POSITION_WEAK = 10.0
CTR_LOW = df.loc[df["impressions"] >= IMPRESSIONS_VISIBLE, "ctr"].median()

df["flag_visible"] = (df["impressions"] >= IMPRESSIONS_VISIBLE).astype(int)
df["flag_weak_position"] = (df["avg_position"] >= POSITION_WEAK).astype(int)
df["flag_low_ctr"] = (df["ctr"] < CTR_LOW).astype(int)
df["baseline_score"] = df["flag_visible"] * (df["flag_weak_position"] + df["flag_low_ctr"]) * df["impressions"]

def reason_code(row):
    if not row["flag_visible"]:
        return "not_visible_enough"
    codes = []
    if row["flag_weak_position"]: codes.append("visible_weak_position")
    if row["flag_low_ctr"]: codes.append("visible_low_ctr")
    return "|".join(codes) if codes else "general_refresh_review"

df["reason_code"] = df.apply(reason_code, axis=1)
ranked_baseline = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked_baseline["rank"] = ranked_baseline.index + 1

p50 = precision_at_k(ranked_baseline["future_decline"], ranked_baseline["baseline_score"], 50)
p100 = precision_at_k(ranked_baseline["future_decline"], ranked_baseline["baseline_score"], 100)
base = base_rate(ranked_baseline["future_decline"])

baseline_metrics = {
    "decision_month": "2026-03", "outcome_month": "2026-04",
    "n_rows": int(len(ranked_baseline)), "base_rate": base,
    "precision_at_50": p50, "precision_at_100": p100,
    "thresholds": {"impressions_visible": float(IMPRESSIONS_VISIBLE), "position_weak": POSITION_WEAK, "ctr_low": float(CTR_LOW)},
}
write_json(OUT_DIR / "baseline_metrics.json", baseline_metrics)
print(f"Base rate: {base:.3f}  Baseline Precision@50: {p50:.3f}  Precision@100: {p100:.3f}")
ranked_baseline.head(20)[["rank","client_hash_id","content_hash_id","baseline_score","reason_code","future_decline"]]

## 4. Model — split design, training, comparison to baseline (same split)

**Split:** grouped by `client_hash_id` (client holdout) — a client's pages
never appear in both train and test, so the model can't partly memorize a
client's baseline traffic level instead of learning a transferable pattern.

**Method:** Logistic Regression as an interpretable reference, Random Forest
as the final model — both trained on the same 8 decision-time features,
compared to the baseline rule on the exact same held-out client split.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

FEATURES = FEATURE_COLUMNS
TARGET = "future_decline"

train_df, test_df = group_holdout_split(dev_frame, "client_hash_id", test_size=0.2, seed=42)
overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
assert len(overlap) == 0, "Client holdout split leaked."
print(f"Train rows: {len(train_df)}  Test rows: {len(test_df)}  Client overlap: {len(overlap)}")

X_train, y_train = train_df[FEATURES].fillna(0), train_df[TARGET]
X_test, y_test = test_df[FEATURES].fillna(0), test_df[TARGET]

scaler = StandardScaler().fit(X_train)
logit = LogisticRegression(max_iter=1000, random_state=42).fit(scaler.transform(X_train), y_train)
logit_scores = logit.predict_proba(scaler.transform(X_test))[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# Same baseline RULE, scored on this exact test split for a fair comparison.
IMPRESSIONS_VISIBLE_TR = train_df["impressions"].quantile(0.50)
CTR_LOW_TR = train_df.loc[train_df["impressions"] >= IMPRESSIONS_VISIBLE_TR, "ctr"].median()
baseline_scores_test = (
    (test_df["impressions"] >= IMPRESSIONS_VISIBLE_TR).astype(int)
    * ((test_df["avg_position"] >= 10).astype(int) + (test_df["ctr"] < CTR_LOW_TR).astype(int))
    * test_df["impressions"]
)

model_results = pd.DataFrame([
    {"model": "baseline_rule", "precision_at_50": precision_at_k(y_test, baseline_scores_test, 50)},
    {"model": "logistic_regression", "precision_at_50": precision_at_k(y_test, logit_scores, 50)},
    {"model": "random_forest", "precision_at_50": precision_at_k(y_test, rf_scores, 50)},
])
model_results["base_rate"] = base_rate(y_test)
write_json(OUT_DIR / "model_metrics.json", model_results.to_dict(orient="records"))
model_results

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
svg_bar_chart("Random Forest feature importance (March -> April decline)",
              importances.index.tolist(), importances.values.tolist(),
              FIG_DIR / "feature_importance.svg",
              caption="Higher bars = more weight in the final model's decline predictions.")

test_df = test_df.copy()
test_df["rf_score"] = rf_scores
test_df["predicted_decline"] = (rf_scores >= 0.5).astype(int)
confident_wrong = test_df[(test_df["predicted_decline"] != test_df[TARGET]) & ((rf_scores > 0.85) | (rf_scores < 0.15))]
print(f"{len(confident_wrong)} confident-but-wrong predictions out of {len(test_df)} test rows.")
confident_wrong[["client_hash_id","content_hash_id"] + FEATURES + [TARGET, "rf_score"]].head(10)

## 5. Validation & leakage audit

Compares the honest client-grouped split against a naive random row split
(the gap *is* the finding), then runs a deliberate leakage demonstration:
fold the label into its own features and confirm the score becomes
unrealistically good.

In [ ]:
def evaluate_split(train_d, test_d, label):
    m = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
    m.fit(train_d[FEATURES].fillna(0), train_d[TARGET])
    scores = m.predict_proba(test_d[FEATURES].fillna(0))[:, 1]
    return {"split": label, "base_rate": base_rate(test_d[TARGET]),
            "precision_at_50": precision_at_k(test_d[TARGET], scores, 50),
            "client_overlap": len(set(train_d["client_hash_id"]) & set(test_d["client_hash_id"]))}

rand_train, rand_test = train_test_split(dev_frame, test_size=0.2, random_state=42)
random_result = evaluate_split(rand_train, rand_test, "random_row_split")
grouped_result = evaluate_split(train_df, test_df, "client_holdout_split")
split_comparison = pd.DataFrame([random_result, grouped_result])
write_json(OUT_DIR / "split_comparison.json", split_comparison.to_dict(orient="records"))
print(split_comparison)

checklist = {
    "features_only_from_decision_month": True,
    "no_label_or_sibling_columns_in_features": TARGET not in FEATURES,
    "ids_excluded_from_features": all(c not in FEATURES for c in ["client_hash_id","content_hash_id"]),
    "grouped_split_used_for_reported_results": True,
}
for k,v in checklist.items():
    print(("PASS " if v else "FAIL "), k)
assert all(checklist.values())

leaky_features = FEATURES + [TARGET]
leaky_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
leaky_model.fit(train_df[leaky_features].fillna(0), train_df[TARGET])
leaky_scores = leaky_model.predict_proba(test_df[leaky_features].fillna(0))[:, 1]
p50_leaky = precision_at_k(test_df[TARGET], leaky_scores, 50)
print(f"\nPrecision@50 with the label folded into its own features: {p50_leaky:.3f}  (honest model above, for comparison)")
write_json(OUT_DIR / "leakage_audit.json", {"checklist": checklist, "leaky_precision_at_50": p50_leaky})

## 6. Sealed test — May 2026 → June 2026 (run once)

The final model, retrained on **all** March→April dev data, evaluated once
on a completely untouched decision/outcome pair. This number does not get
used to tune anything further — it's reported as-is, next to the dev number.


In [ ]:
sealed_frame = build_dev_frame("2026-05", "2026-06")
sealed_frame.to_csv(OUT_DIR / "sealed_frame_may_jun.csv", index=False)

final_model = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
final_model.fit(dev_frame[FEATURES].fillna(0), dev_frame[TARGET])
sealed_scores = final_model.predict_proba(sealed_frame[FEATURES].fillna(0))[:, 1]

sealed_result = {
    "decision_month": "2026-05", "outcome_month": "2026-06",
    "base_rate": base_rate(sealed_frame[TARGET]),
    "precision_at_50": precision_at_k(sealed_frame[TARGET], sealed_scores, 50),
    "n_rows": int(len(sealed_frame)),
}
write_json(OUT_DIR / "sealed_test_result.json", sealed_result)
print("SEALED TEST (run once):")
sealed_result

## 7. Ranked recommendations — live June 2026 scoring queue

Forward-looking: scores the most recent fully-available month (June 2026,
decision-window only, no label yet since July isn't in the warehouse) using
the final model, and assigns action categories + reason codes + a no-go
floor for low-volume, noisy items.


In [ ]:
live_frame = aggregate_month_features("2026-06")
live_frame.to_csv(OUT_DIR / "live_frame_jun.csv", index=False)
live_frame["model_score"] = final_model.predict_proba(live_frame[FEATURES].fillna(0))[:, 1]

IMPR_VIS_LIVE = live_frame["impressions"].quantile(0.50)
CTR_LOW_LIVE = live_frame.loc[live_frame["impressions"] >= IMPR_VIS_LIVE, "ctr"].median()

def live_reason_code(row):
    if row["impressions"] < IMPR_VIS_LIVE:
        return "low_visibility_monitor_only"
    codes = []
    if row["avg_position"] >= 10: codes.append("weak_position")
    if row["ctr"] < CTR_LOW_LIVE: codes.append("low_ctr")
    return "|".join(codes) if codes else "general_review"

def action_category(score):
    if score >= 0.66: return "refresh_priority"
    if score >= 0.4: return "monitor"
    return "protect"

live_frame["reason_code"] = live_frame.apply(live_reason_code, axis=1)
live_frame["action"] = live_frame["model_score"].apply(action_category)
live_frame["confidence"] = pd.cut(live_frame["model_score"], [0,0.4,0.66,1.0], labels=["low","medium","high"])

NOISE_FLOOR = 20
live_frame.loc[live_frame["impressions"] < NOISE_FLOOR, "action"] = "insufficient_data_manual_review"

ranked_queue = live_frame.sort_values("model_score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

export_cols = ["rank","client_hash_id","content_hash_id","model_score","action","reason_code","confidence",
               "impressions","clicks","avg_position","ctr"]
ranked_queue[export_cols].to_csv(OUT_DIR / "ranked_recommendations.csv", index=False)

action_counts = ranked_queue["action"].value_counts()
svg_bar_chart("Action-category mix, June 2026 live scoring",
              action_counts.index.tolist(), action_counts.values.tolist(),
              FIG_DIR / "action_mix.svg",
              caption="Counts of content items placed into each action category by the final model.")

write_json(OUT_DIR / "action_playbook_summary.json", {
    "decision_month": "2026-06", "n_scored": int(len(ranked_queue)),
    "action_counts": action_counts.to_dict(), "noise_floor_impressions": NOISE_FLOOR,
})
print(action_counts)
ranked_queue.head(20)[export_cols]

## 8. Zip everything for download

Only small, safe artifacts — hashed IDs, aggregated numbers, JSON metrics,
SVG charts. No raw per-day rows, no client names, no credentials.


In [ ]:
import shutil
zip_path = shutil.make_archive("/content/capstone_outputs", "zip", OUT_DIR)
print("Download this file from the Colab file browser (left sidebar):")
print(zip_path)
print("\nThen send it back (or its contents) so the write-up and paper can be built on real numbers.")

## Self-check before you send this back

- [ ] Ran top to bottom with no errors (Runtime → Run all)
- [ ] `capstone_outputs.zip` downloaded from `/content/`
- [ ] No client names, private URLs, or credentials appear anywhere in the printed output
- [ ] Sealed test (section 6) was run exactly once and not re-tuned afterward
